In [1]:
import pandas as pd
from tqdm import tqdm
import time
import os
import json
import anthropic
from collections import Counter
from pathlib import Path


with open("Claude_API.txt", "r") as file:
    api_key = file.read().strip()

os.environ['ANTHROPIC_API_KEY'] = api_key

config = {}

with open("File_Directory.txt", "r") as file:
    for line in file:
        if ":" in line:
            key, value = line.split(":", 1)
            config[key.strip()] = value.strip()

input_file = Path(config["Input"])
output_dir = Path(config["Output"])

# UPDATE
INPUT_FILE = input_file

# UPDATE
OUTPUT_DIR = output_dir

objective_columns = [
    'PRIIPS KID Objective',
    'KIID Objective/Investment Policy',
    'Prospectus Objective',
    'Investment Strategy - English',
    'PRIIPS KID Objective - Danish',
    'PRIIPS KID Objective - Dutch',
    'PRIIPS KID Objective - Finnish',
    'PRIIPS KID Objective - French',
    'PRIIPS KID Objective - German',
    'PRIIPS KID Objective - Italian',
    'PRIIPS KID Objective - Norwegian',
    'PRIIPS KID Objective - Portuguese',
    'PRIIPS KID Objective - Spanish',
    'PRIIPS KID Objective - Swedish',
    'KIID Objective/Investment Policy - German',
    'KIID Objective/Investment Policy - French',
    'KIID Objective/Investment Policy - Italian',
    'KIID Objective/Investment Policy - Spanish',
    'KIID Objective/Investment Policy - Norwegian',
    'KIID Objective/Investment Policy - Swedish',
    'KIID Objective/Investment Policy - Finnish',
    'KIID Objective/Investment Policy - Portuguese',
    'KIID Objective/Investment Policy - Danish',
    'Investment Strategy - Danish',
    'Investment Strategy - Finnish',
    'Investment Strategy - French',
    'Investment Strategy - German',
    'Investment Strategy - Italian',
    'Investment Strategy - Norwegian',
    'Investment Strategy - Portuguese',
    'Investment Strategy - Spanish',
    'Investment Strategy - Swedish',
    'Strategy Description'
]

In [ ]:
SYSTEM_PROMPT = """You are extracting the fund objective or objectives from regulatory disclosure text for a European mutual fund.

WHAT IS A FUND OBJECTIVE:
The fund objective is the statement of what the fund aims to achieve for its investors — its goal or intended outcome.

Examples of objectives include:
- Long-term capital growth
- Long-term value increase
- Regular income
- Maximizing total returns
- Beating an index or benchmark
- Matching the performance of an index

MULTIPLE OBJECTIVES:
There may be more than one objective. Extract ALL objectives and label them separately. Examples of multiple objectives include:
- The highest possible increase in value and a broad spread of risks
- To increase the value of your investment and to outperform the equity market
- To outperform the benchmark index while maintaining lower carbon intensity than the index
- Long-term capital appreciation with a higher ESG score than the index

If a single sentence contains two objectives, always split them into separate objectives, even if they are both financial or both sustainable objectives. Examples:
- "provide income and moderate capital growth" → Objective 1: "provide income", Objective 2: "provide moderate capital growth"
- "generate good returns which also exceed the benchmark index" → Objective 1: "generate good returns", Objective 2: "exceed the benchmark index"
- "seek financial profitability and alignment with the Sustainable Development Goals as defined by the UN" → Objective 1: "seek financial profitability", Objective 2: "seek alignment with the Sustainable Development Goals as defined by the UN"
- "generating both social returns and long-term capital growth" → Objective 1: "generating social returns", Objective 2: "generating long-term capital growth"
- "exceed the performance of the index while maintaining a higher ESG score than the index" → Objective 1: "exceed the performance of the index", Objective 2: "maintaining a higher ESG score than the index"
- "seek long-term capital growth while reducing the risk of capital loss" → Objective 1: "seek long-term capital growth", Objective 2: "reducing the risk of capital loss"
- "achieve a net performance after fees superior to the index, with the intention of generating an indirect impact on employment" → Objective 1: "achieve a net performance after fees superior to the index", Objective 2: "generating an indirect impact on employment"
- "offer the highest possible growth of capital in combination with broad risk diversification" → Objective 1: "offer the highest possible growth of capital", Objective 2: "broad risk diversification"

Splitting guidance:
- Split objectives not only when joined by "and" or "while", but also when the second objective is introduced by wording such as "which also", "that also", or similar continuation phrases.
- If one part states a financial goal and another part states a sustainability goal, extract them as separate objectives.
- Treat formulations like "while maintaining a higher ESG score than the index" or "while maintaining lower carbon intensity than the index" as a separate objective, not as a discardable constraint.

There may be both financial objectives and sustainability objectives. Examples of sustainability objectives include:
- Reducing greenhouse gas emissions
- Increasing biodiversity
- Improving living standards
- Advancing the UN Sustainable Development Goals (SDGs)

DO NOT INCLUDE - INVESTMENT POLICY AND STRATEGY:
- What the fund invests in: asset allocation, geographic or sector focus, types of securities
- How securities are selected: active vs passive approach, screening methodology
- The mechanism through which an objective is achieved. These must be removed from the objective text, even if it appears in the same sentence as the objective. Example: "provide investors with an appreciation of the invested capital, obtaining gains through the appreciation of European companies' stock" → extract only "provide investors with an appreciation of the invested capital"

DO NOT INCLUDE – TYPES OF COMPANIES THE FUND INVESTS IN:
- Any descriptions of or restrictions on the types of companies the fund invests in. Examples:
  - "advance the UN SDGs by investing in companies whose products contribute to SDGs" → extract only "advance the UN SDGs"
  - "generate an impact on employment by investing in companies that contribute to improving employment" → extract only "generate an impact on employment"
  - "aims to achieve long-term capital growth by investing in high quality companies. These companies should make a positive contribution to sustainable development within the countries in which they operate." → extract only "long-term capital growth"
- If the text describes what the companies do, rather than what the fund itself aims to achieve, exclude that wording.
- Descriptions such as "invest in companies that...", "companies whose products...", "companies that contribute to...", "companies that support...", "issuers contributing to...", or similar formulations describe the types of companies the fund invests in, not the fund's objective.
- If an action or outcome appears only inside a description of what companies do or how they qualify for inclusion, do NOT extract it as an objective.
- Even if the sentence explicitly says "sustainable investment objective", extract only the fund's own intended outcome, not company activities or issuer characteristics.

DO NOT INCLUDE – OTHER INFORMATION TAKEN INTO ACCOUNT:
Any description of other information taken into account when pursuing the objective, such as "while taking into account Environmental, Social and Governance (ESG) criteria" or "taking into account the risk level".

DO NOT INCLUDE - REGULATORY BOILERPLATE:
Any statement that the fund "promotes environmental and/or social characteristics" or "is promoting ESG characteristics" is required boilerplate language under Article 8/9 of the EU's SFDR regulation. It is not a fund-specific objective and must never be extracted as one, even if it appears in the objectives section of the document. However, if the fund promotes environmental, social, or governance outcomes (e.g., "reduced carbon emissions" or "good governance"), this is part of the objective and should be extracted as one.

DO NOT INCLUDE - OTHER:
- Risk information
- Distribution/dividend policy
- Benchmark details beyond a simple reference, if the objective is to track or outperform it
- References to benchmarks solely for performance comparison (e.g. "management takes as reference the profitability of the index, solely for informational or comparative purposes" is not an objective)
- Do not extract the same objective twice. If the same goal is restated in different sentences or columns, extract it only once.
- If a sentence repeats a previously extracted financial objective and adds a sustainability objective, extract only the new sustainability objective as a separate objective.

TIME HORIZON:
If the objective includes a time horizon or investment period, always include it as part of the objective text. Example: "outperform the index over a rolling five-year period" → include "over a rolling five-year period".

EXTRACTION RULES:
1. Extract ONLY the fund objective(s)
2. Extract text VERBATIM - copy the exact words from the source, do not paraphrase
3. If there are multiple objectives, separate them as Objective_1, Objective_2, Objective_3 etc.
4. Classify each objective as "financial" or "sustainable"
5. Return concise statements that directly replicate the text from the regulatory filing
6. If the objective cannot be identified, return "NOT IDENTIFIED"

YOU WILL BE PROVIDED:
Multiple columns containing objective-related text, potentially in multiple languages. Select the column with the clearest, most complete objective statement.

SOURCE SELECTION CRITERIA:
- Clarity: Does it explicitly state "objective", "aim", or "seeks"?
- Completeness: Does it include the full objective statement?
- Conciseness: Is it focused on objectives vs. mixed with strategy/policy?
- Language: Prefer English if quality is equal; if the English column lacks a clear objective or is unavailable, use the non-English column even if translation is required


TRANSLATION:
- Some columns may contain text in non-English languages (French, German, Norwegian, Spanish, Swedish, Danish, etc.)
- If the best source column is in a non-English language, translate it to English
- Include the full translated text of the entire source column in a field called "translated_source_value" — not just the objective sentence(s), but the complete column text
- When translating, use natural English phrasing rather than literal translation (e.g., "la recherche de performance" → "seeks to achieve performance", not "the search for performance")
- All extracted objective_text values must be in English
- Record the language of the source column in the "source_language" field, regardless of whether translation was needed

OUTPUT FORMAT:
Return a JSON object with this structure:
{
  "selected_source_column": "exact column name from input",
  "source_language": "English" or "French" or "German" etc.,
  "translated_source_value": "full translated text of entire source column (only if translation was needed, otherwise null)",
  "source_selection_reason": "brief explanation of why this column was chosen",
  "objectives": [
    {
      "objective_number": 1,
      "objective_text": "EXACT TEXT copied verbatim from source - fund objective only, not company activities",
      "objective_type": "financial" or "sustainable"
    }
  ],
  "confidence": "high" or "medium" or "low" or "none",
  "notes": "any relevant notes"
}

Confidence levels:
- "high": Clear objective statement found, extracted verbatim with no ambiguity
- "medium": Objective found but mixed with other content or requires interpretation
- "low": Objective statement unclear or potentially incomplete
- "none": No objective statement found - return "NOT IDENTIFIED"

If NO objective can be found, return:
{
  "selected_source_column": null,
  "source_language": null,
  "source_selection_reason": "No clear objective statement in any column",
  "objectives": [],
  "confidence": "none",
  "notes": "NOT IDENTIFIED"
}
"""
'''Removed: 
Important source-selection guidance:
If one column contains a clear financial objective and another column contains a clear sustainability objective, prefer the column with the clearest and most complete objective statement overall.
If objectives are genuinely split across columns, select the clearest source column and note any limitation in "notes" rather than inventing an objective from unclear text.
'''


# ============================================================================
# FEW-SHOT EXAMPLES
# ============================================================================

FEW_SHOT_EXAMPLES = [
    {
        "fund_name": "Candriam Sustainable Emerging Markets",
        "columns": {
            "PRIIPS KID Objective": "Principal assets traded: Shares of companies with their registered office and/or their principal activities in the emerging countries. Investment strategy: The fund seeks to achieve capital growth by investing in the principal assets traded and to outperform the benchmark. The management team makes discretionary investment choices on the basis of an economic/financial analysis process as well as on a Candriam's proprietary analysis of Environmental, Social and Governance (ESG) considerations. The fund's sustainable investment objective is to contribute to reducing greenhouse gas emissions through specific targets as well as the integration of climate related indicators in issuer and securities analysis. The fund also aims to have long-term positive impact on environment and social objectives.",
            "KIID Objective/Investment Policy": "Not available"
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Only column with content; explicitly states 'the fund seeks' and 'sustainable investment objective is'; contains both financial and sustainable objectives clearly separated",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "achieve capital growth",
                    "objective_type": "financial"
                },
                {
                    "objective_number": 2,
                    "objective_text": "outperform the benchmark",
                    "objective_type": "financial"
                },
                {
                    "objective_number": 3,
                    "objective_text": "contribute to reducing greenhouse gas emissions",
                    "objective_type": "sustainable"
                },
                {
                    "objective_number": 4,
                    "objective_text": "have long-term positive impact on environment and social objectives",
                    "objective_type": "sustainable"
                }
            ],
            "confidence": "high",
            "notes": "Fund clearly states two distinct financial objectives and two distinct sustainable objectives; investment strategy and ESG analysis process excluded as per instructions"
        }
    },
    {
        "fund_name": "M&G Pan European",
        "columns": {
            "KIID Objective/Investment Policy": "The fund aims to: financial objective: provide a combination of capital growth and income to deliver a higher return than the European stock market over any five-year period; and sustainable objective: invest in companies that contribute towards the Paris Agreement climate change goal."
        },
        "response": {
            "selected_source_column": "KIID Objective/Investment Policy",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Only column with content; explicit financial objective is present, but the purported sustainable objective is phrased only as a company-selection criterion",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "provide a combination of capital growth and income to deliver a higher return than the European stock market over any five-year period",
                    "objective_type": "financial"
                }
            ],
            "confidence": "high",
            "notes": "Only the financial objective is extracted. 'invest in companies that contribute towards the Paris Agreement climate change goal' is excluded because it describes the types of companies invested in, not a fund-level outcome"
        }
    },
    {
        "fund_name": "Danske Invest Japani Osake K",
        "columns": {
            "PRIIPS KID Objective": "The target of investment activities is to achieve long-term value increase on the invested assets through active asset management pursuant to the Act on Common Funds and the fund's regulations. The assets are invested in publicly traded equities and equity-linked securities of companies listed in Japan. The objective is, through equity selection, to achieve a return that outperforms the average return on the Japanese equity market. The fund is categorised as article 8 under SFDR and promotes environmental and/or social characteristics, as well as good governance practices, through screening, exclusions, investment analysis and decision-making as well as active ownership."
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Only column with content; contains two distinct objective statements clearly separated across sentences",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "achieve long-term value increase on the invested assets",
                    "objective_type": "financial"
                },
                {
                    "objective_number": 2,
                    "objective_text": "achieve a return that outperforms the average return on the Japanese equity market",
                    "objective_type": "financial"
                }
            ],
            "confidence": "high",
            "notes": "Two financial objectives found across separate sentences; active management wording, equity-selection mechanism, and Article 8/SFDR language excluded"
        }
    },
    {
        "fund_name": "GS Central Europe EQ-P Cap CZK",
        "columns": {
            "PRIIPS KID Objective": "The Fund is classified as a financial product under Article 8 of the EU Sustainable Finance Disclosure Regulation. The Fund promotes environmental or social characteristics but does not have a sustainable investment objective. The objective of the fund is to realize long-term capital growth by investing mainly in a diversified portfolio of equities listed and traded on regulated Central European exchanges. Measured over a period of several years we aim to beat the performance of the benchmark 45% Warsaw (WIG30) TR, 35% Prague (PX) TR, 10% Budapest (BUX) TR and 10% Bucharest (BET) TR."
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Only column with content; contains explicit 'objective of the fund is' statement plus a secondary benchmark-beating objective",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "realize long-term capital growth",
                    "objective_type": "financial"
                },
                {
                    "objective_number": 2,
                    "objective_text": "beat the performance of the benchmark 45% Warsaw (WIG30) TR, 35% Prague (PX) TR, 10% Budapest (BUX) TR and 10% Bucharest (BET) TR",
                    "objective_type": "financial"
                }
            ],
            "confidence": "high",
            "notes": "Primary objective explicitly flagged with 'the objective of the fund is'; benchmark-beating objective extracted from secondary sentence; Article 8 boilerplate excluded"
        }
    },
    {
        "fund_name": "BGF World Financials A2",
        "columns": {
            "PRIIPS KID Objective": "The Fund aims to maximise the return on your investment through a combination of capital growth and income on the Fund's assets and invest in a manner consistent with the principles of environmental, social and governance (ESG) investing. The Fund invests globally at least 70% of its total assets in the equity securities (e.g. shares) of companies the main business of which is financial services."
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Only column with content; objective clearly stated with 'the Fund aims to maximise'; the return formulation is part of the objective, while the ESG phrase is an approach description",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "maximise the return on your investment through a combination of capital growth and income",
                    "objective_type": "financial"
                }
            ],
            "confidence": "high",
            "notes": "'and invest in a manner consistent with the principles of environmental, social and governance (ESG) investing' excluded as investment approach rather than objective"
        }
    },
    {
        "fund_name": "BNP Paribas B Inv Glob Eq D Cl Dis",
        "columns": {
            "PRIIPS KID Objective": "The objective of the fund is to provide the highest possible increase in value and to ensure a broad diversification of risks by investing at least 90% of its assets in all types of global shares, any income from which is eligible for deduction in the context of final taxable income (DBI). This sub-fund promotes environmental and social attributes, focusing on social, environmental and governance issues through the application of exclusions and through ESG integration."
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Only column with content; single sentence contains two conjoined objectives linked by 'and'",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "provide the highest possible increase in value",
                    "objective_type": "financial"
                },
                {
                    "objective_number": 2,
                    "objective_text": "ensure a broad diversification of risks",
                    "objective_type": "financial"
                }
            ],
            "confidence": "high",
            "notes": "Two objectives separated from a single conjoined sentence; investment mechanism and ESG language excluded"
        }
    },
    {
        "fund_name": "Deka-ESG Aktien Deutschland CF",
        "columns": {
            "PRIIPS KID Objective": "Objectives: The investment objective of this fund is to generate medium to long-term capital growth by exploiting opportunities and avoiding risks arising from economic, ecological and social developments. The aim is to achieve the greatest possible return while at the same time maintaining appropriate risk for this asset class. To achieve this, the fund invests primarily in shares of companies from Germany or companies included in German stock indices."
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Only column with content; two objective statements are present, and the second sentence contains two distinct objectives joined by 'while'",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "generate medium to long-term capital growth",
                    "objective_type": "financial"
                },
                {
                    "objective_number": 2,
                    "objective_text": "achieve the greatest possible return",
                    "objective_type": "financial"
                },
                {
                    "objective_number": 3,
                    "objective_text": "maintaining appropriate risk for this asset class",
                    "objective_type": "financial"
                }
            ],
            "confidence": "high",
            "notes": "The first sentence contains one financial objective. The second sentence contains two financial objectives joined by 'while'. The mechanism introduced by 'by exploiting...' and the subsequent investment strategy text are excluded"
        }
    },
    {
        "fund_name": "ICA Banken Modig",
        "columns": {
            "PRIIPS KID Objective": "Fondens mål är att ge en avkastning som över tid överstiger fondens sammansatta jämförelseindex (från och med 2019-01-01), 35 procent Six Return Index (SIXRX) och 65 procent MSCI AC World. Fonden är en aktivt förvaltad fond."
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective",
            "source_language": "Swedish",
            "translated_source_value": "The fund's objective is to generate a return over time that exceeds the fund's composite benchmark index (as of 2019-01-01), 35 percent Six Return Index (SIXRX) and 65 percent MSCI AC World. The fund is actively managed.",
            "source_selection_reason": "Only column with content; written in Swedish but contains a clear objective statement",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "generate a return over time that exceeds the fund's composite benchmark index (as of 2019-01-01), 35 percent Six Return Index (SIXRX) and 65 percent MSCI AC World",
                    "objective_type": "financial"
                }
            ],
            "confidence": "high",
            "notes": "Source text in Swedish; extracted objective translated into English; active management wording excluded"
        }
    },
    {
        "fund_name": "Caixabank Bolsa Gestión Euro Estándar FI",
        "columns": {
            "PRIIPS KID Objective": "Management objective: Management takes as reference the profitability of the EUROSTOXX 50 Index (Total Return), solely for informational or comparative purposes. Investment policy: Will invest more than 75% of total exposure in equity assets of European issuers/markets."
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Only column with content; however the text does not contain a clear objective statement — using the index as a reference for informational purposes is not equivalent to aiming to outperform it",
            "objectives": [],
            "confidence": "none",
            "notes": "NOT IDENTIFIED — the text references a benchmark for comparative purposes only, which does not constitute a fund objective; no clear aim or goal stated"
        }
    },
    {
        "fund_name": "Janus Henderson Hrzn US Ss Eq GU2USD",
        "columns": {
            "PRIIPS KID Objective": "The Fund aims to provide capital growth over the long term (5 years or more) by investing in US companies whose products and services are considered by the investment manager as contributing to positive environmental or social change and thereby have an impact on the development of a sustainable global economy."
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Only column with content; clear financial objective is stated, while the remainder of the sentence describes the investment mechanism and company characteristics",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "provide capital growth over the long term (5 years or more)",
                    "objective_type": "financial"
                }
            ],
            "confidence": "high",
            "notes": "Only the financial objective is extracted. The clause beginning 'by investing in US companies whose products and services...' is excluded because it describes company attributes and mechanism, not a separate fund-level sustainable objective"
        }
    },
    {
        "fund_name": "HSBC GIF Global Eq Qual Inc ZQ1",
        "columns": {
            "PRIIPS KID Objective": "Investment Objective: The Fund aims to provide income and moderate capital growth by investing in global developed market shares, while promoting environmental, social and governance (ESG) characteristics. The Fund aims to do this with a lower carbon intensity, calculated as a weighted average of the carbon intensities of the Fund's investments, than the weighted average of the constituents of the MSCI World Index."
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Only column with content; the first sentence contains two conjoined financial objectives and the second sentence states a separate carbon-intensity objective",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "provide income",
                    "objective_type": "financial"
                },
                {
                    "objective_number": 2,
                    "objective_text": "provide moderate capital growth",
                    "objective_type": "financial"
                },
                {
                    "objective_number": 3,
                    "objective_text": "lower carbon intensity, calculated as a weighted average of the carbon intensities of the Fund's investments, than the weighted average of the constituents of the MSCI World Index",
                    "objective_type": "sustainable"
                }
            ],
            "confidence": "high",
            "notes": "'promoting environmental, social and governance (ESG) characteristics' is excluded as regulatory boilerplate; the lower-carbon-intensity clause is retained as a separate sustainability objective"
        }
    },
    {
        "fund_name": "Maj Invest Value Aktier Akk",
        "columns": {
            "PRIIPS KID Objective - Danish": "Målet med denne afdeling er at opnå et afkast, som mindst svarer til den generelle udvikling for globale aktier. Endvidere har afdelingen et ønske om at fremme miljømæssige og/eller sociale forhold i overensstemmelse med artikel 8 i EU's SFDR."
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective - Danish",
            "source_language": "Danish",
            "translated_source_value": "The objective of this share class is to achieve a return that at least corresponds to the general development of global equities. Furthermore, the share class wishes to promote environmental and/or social characteristics in accordance with Article 8 of the EU's SFDR.",
            "source_selection_reason": "Only column with content; written in Danish; first sentence contains the fund objective",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "achieve a return that at least corresponds to the general development of global equities",
                    "objective_type": "financial"
                }
            ],
            "confidence": "high",
            "notes": "Source in Danish; 'promote environmental and/or social characteristics in accordance with Article 8 of the EU's SFDR' excluded as SFDR boilerplate, not a fund-specific objective"
        }
    },
    {
        "fund_name": "Mirova US Sustainable Equity I/A USD",
        "columns": {
            "KIID Objective/Investment Policy": "The Sub-Fund has a sustainable investment objective which is to allocate the capital towards sustainable economic models with environmental and/or social benefits by investing in companies which qualify as a sustainable investment and whose economic activity contributes positively to or does not significantly harm the achievement of one or more of the UN Sustainable Development Goals (SDGs) and/or reduces the risk of not achieving one or more of the UN SDGs, while ensuring that the portfolio companies follow good governance practices. The Sub-Fund will seek to invest in US domiciled companies, listed on the US stock exchanges, while systematically including Environmental, Social and Governance (''ESG'') considerations with financial performance measured against the S&P 500 Net Dividends Reinvested Index over the recommended minimum investment period of 5 years."
        },
        "response": {
            "selected_source_column": "KIID Objective/Investment Policy",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Most concise and explicit objective statement; the opening sentence clearly states a sustainable investment objective, while the remainder consists of mechanism, company qualification, governance, and policy details",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "allocate the capital towards sustainable economic models with environmental and/or social benefits",
                    "objective_type": "sustainable"
                }
            ],
            "confidence": "high",
            "notes": "Only the fund-level sustainable outcome is extracted. The clause beginning 'by investing in companies which qualify...' and the SDG contribution wording are excluded because they describe company qualification and company activities rather than a separate fund objective"
        }
    },
    {
        "fund_name": "FIRST Impact",
        "columns": {
            "PRIIPS KID Objective": "Målsetting\n\nFondets målsetting er å skape høyest mulig relativ avkastning mot referanseindeksen, MSCI World AC, Net Total Return (målt i NOK).\n\nFondet skal investere i selskaper globalt som har løsninger på FN’s bærekraftsmål og dermed bidrar til omstillingen til et mer bærekraftig samfunn. Fondet har en aktiv investeringsstrategi og vesentlige avvik fra referanseindeksen må påregnes. Fondet søker å identifisere de sektorene og temaene som vil gi best eksponering til god vekst og lønnsomhet ved å løse noen av FN’s bærekraftsmål, samt identifisere de beste investeringsmulighetene innenfor disse temaene."
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective",
            "source_language": "Norwegian",
            "translated_source_value": "Objective\n\nThe fund's objective is to create the highest possible relative return against the benchmark index, MSCI World AC, Net Total Return (measured in NOK).\n\nThe fund will invest in companies globally that have solutions to the UN's sustainability goals and thereby contribute to the transition to a more sustainable society. The fund has an active investment strategy and significant deviations from the benchmark index must be expected. The fund seeks to identify the sectors and themes that will provide the best exposure to good growth and profitability by solving some of the UN's sustainability goals, as well as identify the best investment opportunities within these themes.",
            "source_selection_reason": "Most complete source with a clear objective section; only the first sentence states the fund's own objective directly",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "create the highest possible relative return against the benchmark index, MSCI World AC, Net Total Return (measured in NOK)",
                    "objective_type": "financial"
                }
            ],
            "confidence": "high",
            "notes": "Only the financial objective is extracted. The sustainability language is excluded because it appears only in clauses describing the types of companies the fund invests in and what those companies contribute to"
        }
    },
    {
        "fund_name": "Example Fund With Growth And Downside Objective",
        "columns": {
            "PRIIPS KID Objective": "The fund seeks long-term capital growth while reducing the risk of capital loss."
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Clear English objective sentence containing two distinct financial objectives: one return/growth objective and one risk-control objective",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "seeks long-term capital growth",
                    "objective_type": "financial"
                },
                {
                    "objective_number": 2,
                    "objective_text": "reducing the risk of capital loss",
                    "objective_type": "financial"
                }
            ],
            "confidence": "high",
            "notes": "The sentence contains two separate financial objectives joined by 'while': a growth objective and a downside-risk objective"
        }
    },
    {
        "fund_name": "Lannebo Småbolag A",
        "columns": {
            "KIID Objective/Investment Policy": "The fund invests in equities of small and medium-sized listed companies in the Nordic region with a focus on Sweden. The companies’ market capitalization at the time of investment may not exceed 1 per cent of the Swedish stock market’s total market capitalization. At the end of 2020 this amounted to a market capitalization of around SEK 88bn. The fund does not invest in companies that violate international conventions or in companies that produce and/or distribute controversial weapons. The fund also refrains from investing in companies in which more than 5 per cent of the company’s turnover comes from the production and/or distribution of tobacco, cannabis, alcohol, weapons, gambling or pornography. The fund also refrains from investing in companies in which more than 5 per cent of the company’s turnover comes from extracting of fossil fuels (coal, oil and gas). The fund is managed according to Lannebo Fonder’s investment philosophy of active asset management. This means that we focus on fundamental company analysis and valuation of individual stocks in companies whose business models we understand. We are independent in our management of, and in our decisions regarding, the fund. Focus is placed on each of the fund’s individual investments and its outlooks. The most attractive shares, according to the managers, are the largest holdings in the fund, regardless of the holdings in the fund’s benchmark. We try to avoid factors that affect stock prices in the short term, resulting in that we invest in companies where we may have a qualified opinion about the company prospects where valuations are considered attractive enough. The fund normally invests in 40 to 60 companies. Deviations from the benchmark index can be both larger and smaller, please see the fund’s annual report or information brochure for an account of the fund’s actual deviations in relation to its benchmark over time. The objective is to over time generate good returns which also exceeds the benchmark index taking into account the risk level. The fund normally pays no dividend and all income is reinvested in the fund. This document is valid for share class SEK. Purchase and sale of fund units may normally be made every banking day. Benchmark: Carnegie Small Cap Return Index Sweden. Recommendation: The fund may not be suitable for investors who plan to withdraw their money within five years."
        },
        "response": {
            "selected_source_column": "KIID Objective/Investment Policy",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Most complete and clear objective statement in English; explicitly states the objective in a dedicated sentence",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "over time generate good returns",
                    "objective_type": "financial"
                },
                {
                    "objective_number": 2,
                    "objective_text": "exceeds the benchmark index",
                    "objective_type": "financial"
                }
            ],
            "confidence": "high",
            "notes": "The sentence contains two financial objectives: a general return objective and a benchmark-relative objective. 'taking into account the risk level' is excluded as other information taken into account rather than part of the objective"
        }
    },
    {
        "fund_name": "Example Fund With ESG Qualifier Objective",
        "columns": {
            "PRIIPS KID Objective": "The fund seeks to exceed the performance of the index while maintaining a higher ESG score than the index."
        },
        "response": {
            "selected_source_column": "PRIIPS KID Objective",
            "source_language": "English",
            "translated_source_value": None,
            "source_selection_reason": "Clear English objective sentence containing one financial objective and one sustainability objective joined by 'while'",
            "objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "exceed the performance of the index",
                    "objective_type": "financial"
                },
                {
                    "objective_number": 2,
                    "objective_text": "maintaining a higher ESG score than the index",
                    "objective_type": "sustainable"
                }
            ],
            "confidence": "high",
            "notes": "The sentence contains two separate objectives: a benchmark-relative financial objective and an ESG-related sustainability objective"
        }
    }
]

In [3]:
def build_column_dict(row, objective_columns):
    """
    Build dictionary of available columns for this fund (NO translation here)
    """
    columns_dict = {}
    for col in objective_columns:
        if col in row.index:
            value = row[col]
            if pd.notna(value) and str(value).strip() not in ['-', 'Not available', '']:
                columns_dict[col] = str(value)
            else:
                columns_dict[col] = "Not available"
    return columns_dict
    
def extract_with_claude(fund_name, fund_id, columns_dict):
    columns_text = "\n".join([f"- {col}: {val}" for col, val in columns_dict.items()])
    
    user_prompt = f"""Fund ID: {fund_id}
Fund Name: {fund_name}

Available Columns:
{columns_text}"""

    messages = []

    for example in FEW_SHOT_EXAMPLES:
        ex_cols = "\n".join([f"- {col}: {val}" for col, val in example["columns"].items()])
        messages.append({
            "role": "user",
            "content": f"""Fund Name: {example['fund_name']}

Available Columns:
{ex_cols}"""
        })
        messages.append({
            "role": "assistant",
            "content": json.dumps(example["response"], indent=2)
        })

    messages.append({"role": "user", "content": user_prompt})

    try:
        client = anthropic.Anthropic()
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=2000,
            temperature=0,
            system=SYSTEM_PROMPT,
            messages=messages
        )

        print(f"   [{fund_name}] tokens — input: {response.usage.input_tokens}, output: {response.usage.output_tokens}") # track usage
        
        response_text = response.content[0].text

        try:
            return json.loads(response_text)
        except json.JSONDecodeError:
            if "```json" in response_text:
                json_text = response_text.split("```json")[1].split("```")[0].strip()
                return json.loads(json_text)
            elif "```" in response_text:
                json_text = response_text.split("```")[1].split("```")[0].strip()
                return json.loads(json_text)
            else:
                return {
                    "selected_source_column": None,
                    "source_selection_reason": "Failed to parse response",
                    "objectives": [],
                    "confidence": "none",
                    "notes": f"JSON parse error: {response_text[:200]}"
                }

    except Exception as e:
        print(f"   Error for {fund_name}: {str(e)}")
        return {
            "selected_source_column": None,
            "source_selection_reason": f"API error: {str(e)}",
            "objectives": [],
            "confidence": "none",
            "notes": f"API error: {str(e)}"
        }

In [4]:
print("Loading data...")
try:
    df = pd.read_excel(INPUT_FILE)
    print(f" Data loaded successfully")
    print(f"  Total funds available: {len(df)}")
    print(f"  Columns in dataset: {len(df.columns)}")
except FileNotFoundError:
    print(f" ERROR: File not found at {INPUT_FILE}")
except Exception as e:
    print(f" ERROR loading file: {str(e)}")

Loading data...
 Data loaded successfully
  Total funds available: 5680
  Columns in dataset: 133


In [5]:
print("=" * 100)
print("QUICK TEST: Extracting from 10 random funds")
print("=" * 100)

# Sample 10 random funds
df_test = df.sample(n=10, random_state=0)

test_results = []

for idx in tqdm(range(len(df_test)), desc="Quick Test"):
    fund_id = df_test.iloc[idx]['FundId']
    fund_name = df_test.iloc[idx]['Name']
    
    # Build columns dictionary
    columns_dict = build_column_dict(df_test.iloc[idx], objective_columns)
    
    # Extract with Claude
    claude_result = extract_with_claude(fund_name, fund_id, columns_dict)
    
    # Store results
    test_results.append({
        'Fund_Name': fund_name,
        'Selected_Source': claude_result.get('selected_source_column'),
        'Num_Objectives': len(claude_result.get('objectives', [])),
        'Confidence': claude_result.get('confidence'),
        'Full_Result': claude_result
    })

print("QUICK TEST RESULTS:")

for i, result in enumerate(test_results, 1):
    print(f"\n{i}. {result['Fund_Name']}")
    print(f"   Source: {result['Selected_Source']}")
    print(f"   Objectives: {result['Num_Objectives']}")
    print(f"   Confidence: {result['Confidence']}")
    
    if result['Num_Objectives'] > 0:
        for obj in result['Full_Result']['objectives']:
            print(f"      - ({obj['objective_type']}) {obj['objective_text'][:80]}...")

QUICK TEST: Extracting from 10 random funds


Quick Test:  10%|█         | 1/10 [00:09<01:23,  9.25s/it]

   [Robeco Sustainable Property Eqs D €] tokens — input: 16724, output: 347


Quick Test:  20%|██        | 2/10 [00:17<01:08,  8.52s/it]

   [JPM Global EM Rsh Enh Eq Act ETF USD Acc] tokens — input: 34217, output: 277


...

   [Nordea 1 - Global Stable Equity BI EUR] tokens — input: 19797, output: 243


Quick Test:  90%|█████████ | 9/10 [01:28<00:09,  9.31s/it]

   [Amundi Fds US Eq Div Growth M2 USD C] tokens — input: 12967, output: 318


Quick Test: 100%|██████████| 10/10 [01:33<00:00,  9.35s/it]

   [PrivilEdge Robeco EM Bt Enh USD X1 MA] tokens — input: 13025, output: 189
QUICK TEST RESULTS:

1. Robeco Sustainable Property Eqs D €
   Source: Investment Strategy - English
   Objectives: 3
   Confidence: high
      - (financial) provide long term capital growth...
      - (sustainable) aiming for a better sustainability profile compared to the Benchmark by promotin...
      - (sustainable) an improved environmental footprint compared to the Benchmark...

2. JPM Global EM Rsh Enh Eq Act ETF USD Acc
   Source: Investment Strategy - English
   Objectives: 1
   Confidence: high
      - (financial) achieve a long-term return in excess of the Benchmark by actively investing prim...

3. Loomis Sayles Global Gr Eq I/A USD
   Source: KIID Objective/Investment Policy
   Objectives: 1
   Confidence: high
      - (financial) long-term growth of capital...

4. Acacia Reinverplus Europa FI
   Source: KIID Objective/Investment Policy - Spanish
   Objectives: 0
   Confidence: none

5. DWS Inves

In [6]:
# Extract from 100 Random Funds + Merge with All Relevant Text Columns 

print("=" * 100)
print("FULL TEST: Extracting from 100 random funds")
print("=" * 100)

# Sample 100 random funds
df_sample = df.sample(n=100, random_state=32)

all_results = []

for idx in tqdm(range(len(df_sample)), desc="Processing"):
    fund_id = df_sample.iloc[idx]['FundId']
    fund_name = df_sample.iloc[idx]['Name']
    
    # Build columns dictionary
    columns_dict = build_column_dict(df_sample.iloc[idx], objective_columns)
    
    # Extract with Claude
    claude_result = extract_with_claude(fund_name, fund_id, columns_dict)
    
    # Parse results
    objectives_list = claude_result.get('objectives', [])
    selected_source = claude_result.get('selected_source_column')
    source_reason = claude_result.get('source_selection_reason', '')
    confidence = claude_result.get('confidence', 'none')
    notes = claude_result.get('notes', '')
    translated_source = claude_result.get('translated_source_value', None)
    source_language = claude_result.get('source_language', None)  
    
    # Get source column value
    source_value = None
    if selected_source and selected_source in df_sample.columns:
        source_value = df_sample.iloc[idx][selected_source]
    
    # Store results
    result = {
        'FundId': fund_id,
        'Fund_Name': fund_name,
        'Selected_Source_Column': selected_source,
        'Source_Language': source_language,  # NEW COLUMN
        'Source_Column_Value': source_value,
        'Translated_Source_Value': translated_source,
        'Source_Selection_Reason': source_reason,
        'Number_of_Objectives': len(objectives_list),
        'Extraction_Confidence': confidence,
        'Extraction_Notes': notes
    }
    
    # Add objectives
    for i in range(3):
        if i < len(objectives_list):
            obj = objectives_list[i]
            result[f'Objective_{i+1}'] = obj.get('objective_text', '')
            result[f'Objective_{i+1}_Type'] = obj.get('objective_type', '')
        else:
            result[f'Objective_{i+1}'] = None
            result[f'Objective_{i+1}_Type'] = None
    
    all_results.append(result)
    
    # Rate limiting
    if idx > 0 and idx % 50 == 0:
        time.sleep(0.5)

# Convert to DataFrame
results_df = pd.DataFrame(all_results)

print("\n" + "=" * 100)
print("EXTRACTION COMPLETE - STATISTICS")
print("=" * 100)

# Statistics
total = len(results_df)
extracted = (results_df['Number_of_Objectives'] > 0).sum()
success_rate = (extracted / total) * 100

print(f"\nOVERALL RESULTS:")
print(f"  Total Funds: {total}")
print(f"  Successfully Extracted: {extracted} ({success_rate:.1f}%)")
print(f"  Not Extracted: {total - extracted} ({100-success_rate:.1f}%)")

print(f"\nMULTIPLE OBJECTIVES:")
print(f"  1 objective: {(results_df['Number_of_Objectives'] == 1).sum()} ({(results_df['Number_of_Objectives'] == 1).sum()/total*100:.1f}%)")
print(f"  2 objectives: {(results_df['Number_of_Objectives'] == 2).sum()} ({(results_df['Number_of_Objectives'] == 2).sum()/total*100:.1f}%)")
print(f"  3+ objectives: {(results_df['Number_of_Objectives'] >= 3).sum()} ({(results_df['Number_of_Objectives'] >= 3).sum()/total*100:.1f}%)")

print(f"\nOBJECTIVE TYPES:")
obj_types = []
for col in ['Objective_1_Type', 'Objective_2_Type', 'Objective_3_Type']:
    obj_types.extend(results_df[col].dropna().tolist())

if obj_types:
    type_counts = Counter(obj_types)
    for obj_type, count in type_counts.items():
        print(f"  {obj_type}: {count} ({count/len(obj_types)*100:.1f}%)")

print(f"\nCONFIDENCE LEVELS:")
for conf, count in results_df['Extraction_Confidence'].value_counts().items():
    print(f"  {conf}: {count} ({count/total*100:.1f}%)")

# Check how many were translated
translated_count = results_df['Translated_Source_Value'].notna().sum()
print(f"\nTRANSLATIONS:")
print(f"  Translated from non-English: {translated_count} ({translated_count/total*100:.1f}%)")

# Language breakdown
print(f"\nSOURCE LANGUAGES:")
for lang, count in results_df['Source_Language'].value_counts().items():
    print(f"  {lang}: {count} ({count/total*100:.1f}%)")

# MERGE WITH ALL ORIGINAL TEXT COLUMNS

print("\n" + "=" * 100)
print("MERGING WITH ALL ORIGINAL TEXT COLUMNS")
print("=" * 100)

# Get the 100 fund IDs that were just tested
test_fund_ids = results_df['FundId'].tolist()

# Filter original dataframe to only these 100 funds
df_matched = df[df['FundId'].isin(test_fund_ids)].copy()

# Define all objective text columns to add
text_columns_to_add = [
    'Prospectus Objective',
    'KIID Objective/Investment Policy',
    'PRIIPS KID Objective',
    'Strategy Description',
    'PRIIPS KID Objective - Danish',
    'PRIIPS KID Objective - Dutch',
    'PRIIPS KID Objective - Finnish',
    'PRIIPS KID Objective - French',
    'PRIIPS KID Objective - German',
    'PRIIPS KID Objective - Italian',
    'PRIIPS KID Objective - Norwegian',
    'PRIIPS KID Objective - Portuguese',
    'PRIIPS KID Objective - Spanish',
    'PRIIPS KID Objective - Swedish',
    'KIID Objective/Investment Policy - German',
    'KIID Objective/Investment Policy - French',
    'KIID Objective/Investment Policy - Italian',
    'KIID Objective/Investment Policy - Spanish',
    'KIID Objective/Investment Policy - Norwegian',
    'KIID Objective/Investment Policy - Swedish',
    'KIID Objective/Investment Policy - Finnish',
    'KIID Objective/Investment Policy - Portuguese',
    'KIID Objective/Investment Policy - Danish',
    'Investment Strategy - English',
    'Investment Strategy - Danish',
    'Investment Strategy - Finnish',
    'Investment Strategy - French',
    'Investment Strategy - German',
    'Investment Strategy - Italian',
    'Investment Strategy - Norwegian',
    'Investment Strategy - Portuguese',
    'Investment Strategy - Spanish',
    'Investment Strategy - Swedish'
]

# Keep only columns that exist
existing_columns = [col for col in text_columns_to_add if col in df_matched.columns]
missing_columns = [col for col in text_columns_to_add if col not in df_matched.columns]

if missing_columns:
    print(f"Missing {len(missing_columns)} columns:")
    for col in missing_columns[:5]:  # Show first 5
        print(f"    - {col}")
    if len(missing_columns) > 5:
        print(f"    ... and {len(missing_columns) - 5} more")

# Select FundId + all text columns
df_text_columns = df_matched[['FundId'] + existing_columns]

# Merge with test results
df_combined = results_df.merge(df_text_columns, on='FundId', how='left')

# SAVE BOTH FILES 

timestamp = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M")

# Save test results only
test_only_filename = f'TEST_Claude_Extraction_100_funds_{timestamp}.xlsx'
test_only_path = os.path.join(OUTPUT_DIR, test_only_filename)
results_df.to_excel(test_only_path, index=False, engine='openpyxl')

# Save combined results + all text columns
combined_filename = f'TEST_Claude_Extraction_100_funds_WITH_ALL_TEXT_{timestamp}.xlsx'
combined_path = os.path.join(OUTPUT_DIR, combined_filename)
df_combined.to_excel(combined_path, index=False, engine='openpyxl')

print("\n" + "=" * 100)
print("FILES SAVED")
print("=" * 100)
print(f"\n1. TEST RESULTS ONLY:")
print(f"   Filename: {test_only_filename}")
print(f"   Columns: {len(results_df.columns)}")
print(f"   Location: {OUTPUT_DIR}")

print(f"\n2. TEST RESULTS + ALL TEXT COLUMNS:")
print(f"   Filename: {combined_filename}")
print(f"   Total columns: {len(df_combined.columns)}")
print(f"     - Extraction results: {len(results_df.columns)}")
print(f"     - Original text columns: {len(existing_columns)}")
print(f"   Location: {OUTPUT_DIR}")

print(f"\n" + "=" * 100)
print("TEXT COLUMNS ADDED TO COMBINED FILE:")
print("=" * 100)
for i, col in enumerate(existing_columns, 1):
    has_data = df_combined[col].notna().sum()
    print(f"  {i}. {col:50} ({has_data}/{len(df_combined)} funds have data)")

FULL TEST: Extracting from 100 random funds


Processing:   1%|          | 1/100 [00:07<11:59,  7.26s/it]

   [MS INVF Global Brands Eq Inc Z] tokens — input: 17907, output: 270


Processing:   2%|▏         | 2/100 [00:14<12:09,  7.44s/it]

   [DWS Global Value LD] tokens — input: 15473, output: 253


...

Processing:  99%|█████████▉| 99/100 [16:05<00:10, 10.74s/it]

   [Ethos Aktiefond A Utdelande (SEK)] tokens — input: 11008, output: 724


Processing: 100%|██████████| 100/100 [16:11<00:00,  9.72s/it]

   [Quaero Capital Cullen US Value X USD] tokens — input: 15344, output: 237

EXTRACTION COMPLETE - STATISTICS

OVERALL RESULTS:
  Total Funds: 100
  Successfully Extracted: 98 (98.0%)
  Not Extracted: 2 (2.0%)

MULTIPLE OBJECTIVES:
  1 objective: 66 (66.0%)
  2 objectives: 24 (24.0%)
  3+ objectives: 8 (8.0%)

OBJECTIVE TYPES:
  financial: 122 (88.4%)
  sustainable: 16 (11.6%)

CONFIDENCE LEVELS:
  high: 88 (88.0%)
  medium: 10 (10.0%)
  none: 2 (2.0%)

TRANSLATIONS:
  Translated from non-English: 35 (35.0%)

SOURCE LANGUAGES:
  English: 65 (65.0%)
  French: 14 (14.0%)
  German: 6 (6.0%)
  Danish: 5 (5.0%)
  Spanish: 5 (5.0%)
  Swedish: 3 (3.0%)
  Italian: 1 (1.0%)
  Dutch: 1 (1.0%)

MERGING WITH ALL ORIGINAL TEXT COLUMNS
Missing 1 columns:
    - KIID Objective/Investment Policy - Danish



FILES SAVED

1. TEST RESULTS ONLY:
   Filename: TEST_Claude_Extraction_100_funds_2026-05-16_21-11.xlsx
   Columns: 16
   Location: /Users/dannyhogan/Desktop/Hogan_RA_Work

2. TEST RESULTS + ALL TEXT COLUMNS:
   Filename: TEST_Claude_Extraction_100_funds_WITH_ALL_TEXT_2026-05-16_21-11.xlsx
   Total columns: 48
     - Extraction results: 16
     - Original text columns: 32
   Location: /Users/dannyhogan/Desktop/Hogan_RA_Work

TEXT COLUMNS ADDED TO COMBINED FILE:
  1. Prospectus Objective                               (0/100 funds have data)
  2. KIID Objective/Investment Policy                   (53/100 funds have data)
  3. PRIIPS KID Objective                               (86/100 funds have data)
  4. Strategy Description                               (5/100 funds have data)
  5. PRIIPS KID Objective - Danish                      (23/100 funds have data)
  6. PRIIPS KID Objective - Dutch                       (30/100 funds have data)
  7. PRIIPS KID Objective - Finnish               